[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/sodacore-certified/notebooks/day-03-writing-checks-sodacl.ipynb#scrollTo=aa110001)

---
# Day 3 · Writing Checks in SodaCL — Row Count, Missing Values, and Duplicates
**certified-journeys / sodacore-certified** · Soda Core for Data Quality · Learn Badge

> **Goal for today:** Master the fundamental SodaCL check types — row count, missing values, invalid values, and duplicate detection — and organise checks across multiple tables in a single checks file.

In [ ]:
%pip install -q soda-core-duckdb

## Setup — Synthetic Multi-Table Dataset

We create three tables that represent a simple e-commerce dataset. We deliberately inject quality issues into each table so the checks can surface them:

| Table | Injected issues |
|---|---|
| `customers` | Duplicate `id`, missing `email` |
| `orders` | NULL `amount`, invalid `status` value |
| `products` | Negative `price`, empty `sku` |

In [ ]:
import duckdb, tempfile, pathlib

tmpdir = pathlib.Path(tempfile.mkdtemp())
db_path = str(tmpdir / "shop.duckdb")

conn = duckdb.connect(db_path)

# customers — duplicate id=1, missing email for id=3
conn.execute("""
    CREATE TABLE customers AS SELECT * FROM (VALUES
        (1, 'Alice',  'alice@example.com'),
        (1, 'Alice2', 'alice2@example.com'),
        (2, 'Bob',    'bob@example.com'),
        (3, 'Carol',  NULL)
    ) t(id, name, email)
""")

# orders — NULL amount for order 3, invalid status 'unknown'
conn.execute("""
    CREATE TABLE orders AS SELECT * FROM (VALUES
        (1, 1, 150.00, 'pending'),
        (2, 2, 320.50, 'shipped'),
        (3, 3,   NULL, 'unknown'),
        (4, 1, 200.00, 'delivered')
    ) t(id, customer_id, amount, status)
""")

# products — negative price for id=2, empty sku for id=3
conn.execute("""
    CREATE TABLE products AS SELECT * FROM (VALUES
        (1, 'Widget A', 'SKU-001',  25.00),
        (2, 'Widget B', 'SKU-002', -10.00),
        (3, 'Widget C', '',         50.00)
    ) t(id, name, sku, price)
""")

conn.close()
print("Database created at:", db_path)
print("Tables: customers, orders, products")

## Step 1 · Row Count Checks with Fail and Warn Thresholds

**`row_count`** is the most fundamental SodaCL check. It verifies a table is not empty and has the expected volume.

SodaCL threshold syntax:
```yaml
checks for my_table:
  - row_count > 0              # simple fail: any count <= 0 fails
  - row_count:
      fail: when < 100         # FAIL if count drops below 100
      warn: when < 500         # WARN if count drops below 500
```

Using **both** `fail` and `warn` gives you a two-stage alert — a warning surface before a hard failure.

In [ ]:
from soda.scan import Scan

config_yml = f"""
data_sources:
  shop:
    type: duckdb
    path: "{db_path}"
"""

# Row count checks: simple and dual-threshold
checks_row_count = """
checks for customers:
  - row_count > 0

checks for orders:
  - row_count > 0

checks for products:
  - row_count:
      fail: when < 1
      warn: when < 10
"""

config_path = tmpdir / "conf.yml"
checks_path = tmpdir / "checks_rowcount.yml"
config_path.write_text(config_yml)
checks_path.write_text(checks_row_count)

scan = Scan()
scan.set_data_source_name("shop")
scan.add_configuration_yaml_file(str(config_path))
scan.add_sodacl_yaml_file(str(checks_path))
scan.execute()

print(scan.get_logs_text())

### What just happened?

- All three `row_count` checks **pass** — each table has rows.
- The `products` table has 3 rows: above the fail threshold (1) but below the warn threshold (10), so it produces a **WARN**.
- **`row_count > 0`** is a shorthand; **`fail: when < N`** is the explicit form — both are valid SodaCL.
- The scan logs show each check result: `PASS`, `WARN`, or `FAIL` with the actual measured value.

## Step 2 · Missing Values Checks

**Missing checks** measure NULL (and optionally empty string) presence. Two metrics:

| Metric | Meaning |
|---|---|
| `missing_count(col)` | Absolute count of NULL rows |
| `missing_percent(col)` | Percentage of NULL rows |

```yaml
checks for orders:
  - missing_count(amount) = 0          # zero NULLs allowed
  - missing_percent(email):
      warn: when > 1                    # warn if > 1% missing
      fail: when > 5                    # fail if > 5% missing
```

📖 https://docs.soda.io/soda-cl/missing-metrics.html

In [ ]:
checks_missing = """
checks for customers:
  - missing_count(email) = 0

checks for orders:
  - missing_count(amount) = 0
  - missing_percent(amount):
      warn: when > 5
      fail: when > 20
"""

checks_path.with_name("checks_missing.yml").write_text(checks_missing)

scan2 = Scan()
scan2.set_data_source_name("shop")
scan2.add_configuration_yaml_file(str(config_path))
scan2.add_sodacl_yaml_file(str(tmpdir / "checks_missing.yml"))
scan2.execute()

print(scan2.get_logs_text())

### What just happened?

- **`customers.email`**: 1 NULL out of 4 rows → `missing_count = 1` → **FAIL** (expected 0).
- **`orders.amount`**: 1 NULL out of 4 rows = 25% missing → **FAIL** on both the absolute and percentage checks.
- The percentage check (`25% > 20%`) triggers **FAIL**, not WARN, because 25 exceeds the fail threshold.
- Use `missing_count` for primary keys (must be exactly 0); use `missing_percent` for optional columns where a small rate is acceptable.

## Step 3 · Duplicate Detection

**`duplicate_count`** counts rows where the specified column(s) appear more than once.

```yaml
checks for customers:
  - duplicate_count(id) = 0            # primary key must be unique
  - duplicate_count(email) = 0         # business key must be unique
  - duplicate_count(order_id, product_id) = 0   # composite key
```

📖 https://docs.soda.io/soda-cl/metrics-and-checks.html

In [ ]:
checks_duplicates = """
checks for customers:
  - duplicate_count(id) = 0
  - duplicate_count(name) = 0

checks for products:
  - duplicate_count(sku):
      warn: when > 0
      fail: when > 2
"""

(tmpdir / "checks_dupes.yml").write_text(checks_duplicates)

scan3 = Scan()
scan3.set_data_source_name("shop")
scan3.add_configuration_yaml_file(str(config_path))
scan3.add_sodacl_yaml_file(str(tmpdir / "checks_dupes.yml"))
scan3.execute()

print(scan3.get_logs_text())

### What just happened?

- **`customers.id`**: id=1 appears twice → `duplicate_count = 1` → **FAIL**.
- **`customers.name`**: 'Alice' and 'Alice2' are different strings → PASS (no exact duplicates).
- **`products.sku`**: The empty string `''` appears once for id=3 — not a duplicate of anything, so PASS.
- Duplicate checks count the number of *extra* occurrences, not the total rows with duplicates.

## Step 4 · Invalid Values Checks

**Validity checks** enforce domain constraints on column values.

| Check type | Example |
|---|---|
| `valid_values` | Status must be one of a set |
| `valid_regex` | SKU must match a pattern |
| `valid_min` / `valid_max` | Price must be non-negative |
| `invalid_count` | Count rows failing validity |
| `invalid_percent` | Percentage failing validity |

```yaml
checks for orders:
  - invalid_count(status) = 0:
      valid values: [pending, shipped, delivered, cancelled]
```

📖 https://docs.soda.io/soda-cl/validity-metrics.html

In [ ]:
checks_validity = """
checks for orders:
  - invalid_count(status) = 0:
      valid values: [pending, shipped, delivered, cancelled]

checks for products:
  - invalid_count(price) = 0:
      valid min: 0
  - invalid_count(sku) = 0:
      valid regex: SKU-[0-9]{3}
"""

(tmpdir / "checks_validity.yml").write_text(checks_validity)

scan4 = Scan()
scan4.set_data_source_name("shop")
scan4.add_configuration_yaml_file(str(config_path))
scan4.add_sodacl_yaml_file(str(tmpdir / "checks_validity.yml"))
scan4.execute()

print(scan4.get_logs_text())

### What just happened?

- **`orders.status`**: 'unknown' is not in the valid values list → `invalid_count = 1` → **FAIL**.
- **`products.price`**: `-10.00` violates `valid min: 0` → `invalid_count = 1` → **FAIL**.
- **`products.sku`**: `''` (empty string) does not match `SKU-[0-9]{3}` → `invalid_count = 1` → **FAIL**.
- `valid_values` performs exact string matching; `valid_regex` runs the pattern against each value; `valid_min`/`valid_max` do numeric range checks.

## Step 5 · Multi-Table Checks File

Organise checks for all three tables in a **single checks YAML file**. Each `checks for <table>:` block is independent — a failure in one table does not stop checks on another.

Best practice: one checks file per domain (e.g. `checks_customers.yml`, `checks_orders.yml`) or one file per pipeline stage (`checks_raw.yml`, `checks_staging.yml`). Add multiple files to a single scan with multiple `scan.add_sodacl_yaml_file()` calls.

In [ ]:
checks_all_tables = """
# ── customers ─────────────────────────────────────────────────────────
checks for customers:
  - row_count > 0
  - missing_count(email) = 0
  - duplicate_count(id) = 0

# ── orders ────────────────────────────────────────────────────────────
checks for orders:
  - row_count > 0
  - missing_count(amount) = 0
  - invalid_count(status) = 0:
      valid values: [pending, shipped, delivered, cancelled]

# ── products ──────────────────────────────────────────────────────────
checks for products:
  - row_count > 0
  - invalid_count(price) = 0:
      valid min: 0
  - duplicate_count(sku):
      fail: when > 2
      warn: when > 0
"""

(tmpdir / "checks_all.yml").write_text(checks_all_tables)

scan5 = Scan()
scan5.set_data_source_name("shop")
scan5.add_configuration_yaml_file(str(config_path))
scan5.add_sodacl_yaml_file(str(tmpdir / "checks_all.yml"))
scan5.execute()

print(scan5.get_logs_text())

### What just happened?

- All three table blocks ran in a single scan — Soda executes each `checks for` block sequentially.
- Multiple failures in different tables are all reported; no table is skipped because another failed.
- The combined scan is the production pattern: one scan call, all tables, a single scan log.
- `scan5.get_logs_text()` returns all results; for programmatic inspection use `scan5.get_scan_results()`.

## Step 6 · Programmatic Result Inspection

`scan.get_scan_results()` returns a dict with a structured summary you can use in CI/CD pipelines to decide whether to halt or continue.

In [ ]:
import json

results = scan5.get_scan_results()

# Pretty-print the top-level keys
print("Result keys:", list(results.keys()) if isinstance(results, dict) else type(results))
print()

# Exit code: 0 = all pass, 1 = warnings, 2 = failures
exit_code = scan5.get_exit_code()
print(f"Exit code: {exit_code}  (0=pass, 1=warn, 2=fail)")
print()

# Gating pattern: raise in CI if any check failed
if exit_code == 2:
    print("[GATE] ❌ Data quality FAIL — pipeline should halt")
elif exit_code == 1:
    print("[GATE] ⚠️  Data quality WARN — pipeline may continue but review needed")
else:
    print("[GATE] ✅ All checks passed — pipeline can continue")

### What just happened?

- **`get_exit_code()`** returns `0` (all pass), `1` (warnings only), or `2` (any failure).
- In a CI pipeline: `if scan.get_exit_code() == 2: raise Exception("DQ failed")` blocks the next step.
- In Airflow: `if scan.get_exit_code() == 2: raise AirflowException("Data quality check failed")` marks the task as failed.
- **`get_scan_results()`** provides a full structured dict with per-check outcomes for logging or alerting.

## Challenge

```python
# Challenge: Add a fourth table and extend the checks file.
#
# 1. Add an `order_items` table to the DuckDB database:
#    columns: id, order_id, product_id, quantity, unit_price
#    Inject: quantity = 0 for one row, unit_price < 0 for another
#
# 2. Add a new checks block to checks_all.yml:
#    checks for order_items:
#      - row_count > 0
#      - invalid_count(quantity) = 0:     # quantity must be >= 1
#          valid min: 1
#      - invalid_count(unit_price) = 0:   # price must be non-negative
#          valid min: 0
#      - missing_count(order_id) = 0
#      - missing_count(product_id) = 0
#
# 3. Run the scan and verify the injected issues are caught.
# 4. Use get_exit_code() and print the appropriate gate message.

# Your solution here
```

---
## Day 3 key concepts recap

| Check type | SodaCL syntax | Catches |
|---|---|---|
| Row count | `row_count > 0` | Empty table, data loss |
| Missing values | `missing_count(col) = 0` | NULL / empty values |
| Missing percent | `missing_percent(col): warn: when > 1` | Gradual null creep |
| Duplicates | `duplicate_count(col) = 0` | PK violations, re-loads |
| Valid values | `invalid_count(col) = 0: valid values: [...]` | Enum constraint violations |
| Valid range | `invalid_count(col) = 0: valid min: 0` | Numeric out-of-range |
| Valid regex | `invalid_count(col) = 0: valid regex: ...` | Format violations |
| Multi-table | Multiple `checks for <table>:` blocks | Whole pipeline scan |

> **Tip:** Start every new table with four checks: `row_count > 0`, `missing_count(pk) = 0`, `duplicate_count(pk) = 0`, and one domain-specific validity check. These four catch the most common data quality incidents.

---
## What's next
**Day 4** → Threshold Checks — Valid Ranges, Freshness, and Volume Anomaly Detection. You'll write numeric range assertions, freshness checks on `updated_at` columns, and detect volume anomalies without hard thresholds.

Mark Day 3 complete in your [tracker](../index.html).